# Rating assignment

`credit_tools.rating.assign_rating` maps borrowers to rating buckets given:

- a `borrower_id`, `ml_score`, and observed `defaulted` outcome per borrower, and
- a `rating_scale`: a `{rating: expected_default_rate}` dictionary.

It fits a monotonic realized-default-rate curve against `ml_score` (higher score = higher risk), then maps each borrower's calibrated default probability to the rating with the closest expected default rate. The scale itself is just data — Moody's idealized default rates ship as a bundled resource, but any `{rating: edr}` mapping works.

In [ ]:
import random

from credit_tools.rating import assign_rating
from credit_tools.resources import load_rating_scale

random.seed(0)

## Load a rating scale

`load_rating_scale` reads any bundled `resources/<name>.json` file.

In [ ]:
scale = load_rating_scale("moody")
dict(list(scale.items())[:5])

## Simulate a borrower portfolio

In practice `ml_scores` and `defaulted` come from your model's predictions and observed outcomes. Here we simulate them so the notebook is self-contained.

In [ ]:
n = 2000
borrower_ids = [f"b{i}" for i in range(n)]
ml_scores = [random.random() for _ in range(n)]
defaulted = [random.random() < score for score in ml_scores]

## Assign ratings

In [ ]:
ratings = assign_rating(borrower_ids, ml_scores, defaulted, scale)
{borrower_ids[i]: ratings[borrower_ids[i]] for i in range(5)}

## Check calibration

For each rating, the realized default rate among the borrowers assigned to it should track the target expected default rate from the scale.

In [ ]:
from collections import defaultdict

realized = defaultdict(list)
for i, borrower_id in enumerate(borrower_ids):
    realized[ratings[borrower_id]].append(defaulted[i])

for rating in sorted(realized, key=lambda r: scale[r]):
    outcomes = realized[rating]
    print(
        f"{rating:5s} target_edr={scale[rating]:.4f}  n={len(outcomes):4d}  "
        f"realized_dr={sum(outcomes) / len(outcomes):.4f}"
    )